In [ ]:
%matplotlib tk

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, CheckButtons
from scipy.signal import butter, filtfilt

In [ ]:
INIT_AMP = 1.0
INIT_FREQ = 0.27
INIT_PHASE = 0.0
INIT_MEAN = 0.1
INIT_COV = 0.1
INIT_CUTOFF = 5.0

t = np.linspace(0, 10, 1000)
fs = 1 / (t[1] - t[0])

noise_state = {
    'mean': None,
    'cov': None,
    'values': None
}

def get_noise(mean, cov):
    """Генерує шум тільки якщо змінилися його власні параметри."""
    if noise_state['values'] is None or noise_state['mean'] != mean or noise_state['cov'] != cov:
        noise_state['mean'] = mean
        noise_state['cov'] = cov
        noise_state['values'] = np.random.normal(mean, np.sqrt(max(cov, 0.001)), len(t))
    return noise_state['values']

def harmonic_with_noise(amplitude, frequency, phase, noise_mean, noise_covariance, show_noise):
    """Пункт 1: Функція генерації чистої гармоніки та зашумленого сигналу."""
    pure = amplitude * np.sin(2 * np.pi * frequency * t + phase)
    noise = get_noise(noise_mean, noise_covariance)
    noisy = pure + noise
    return pure, noisy

def apply_filter(data, cutoff, fs_rate):
    """Пункт 7: Низькочастотний фільтр Баттерворта для згладжування шуму."""
    nyq = 0.5 * fs_rate
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0:
        normal_cutoff = 0.99
    b, a = butter(N=5, Wn=normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

fig, ax = plt.subplots(figsize=(10, 6))
plt.subplots_adjust(bottom=0.4)

pure, noisy = harmonic_with_noise(INIT_AMP, INIT_FREQ, INIT_PHASE, INIT_MEAN, INIT_COV, True)
filtered = apply_filter(noisy, INIT_CUTOFF, fs)

line_noisy, = ax.plot(t, noisy, color='orange', alpha=0.8, label='Noisy Signal')
line_pure, = ax.plot(t, pure, color='purple', linewidth=2, label='Pure Harmonic')
line_filtered, = ax.plot(t, filtered, color='blue', linestyle='--', linewidth=2, label='Filtered')

ax.set_ylim(-3, 3)
ax.grid(True)
ax.legend(loc='upper right')
ax.set_title("Гармоніка з шумом та фільтрацією")

ax_amp = plt.axes([0.2, 0.32, 0.55, 0.025])
ax_freq = plt.axes([0.2, 0.28, 0.55, 0.025])
ax_phase = plt.axes([0.2, 0.24, 0.55, 0.025])
ax_mean = plt.axes([0.2, 0.20, 0.55, 0.025])
ax_cov = plt.axes([0.2, 0.16, 0.55, 0.025])
ax_cutoff = plt.axes([0.2, 0.12, 0.55, 0.025])

s_amp = Slider(ax_amp, 'Amplitude', 0.1, 2.0, valinit=INIT_AMP)
s_freq = Slider(ax_freq, 'Frequency', 0.05, 2.0, valinit=INIT_FREQ)
s_phase = Slider(ax_phase, 'Phase', 0.0, 2*np.pi, valinit=INIT_PHASE)
s_mean = Slider(ax_mean, 'Noise Mean', -1.0, 1.0, valinit=INIT_MEAN)
s_cov = Slider(ax_cov, 'Noise Covariance', 0.0, 0.5, valinit=INIT_COV)
s_cutoff = Slider(ax_cutoff, 'Cutoff Freq', 0.1, 15.0, valinit=INIT_CUTOFF)

ax_check = plt.axes([0.8, 0.16, 0.12, 0.1])
check = CheckButtons(ax_check, ['Show Noise'], [True])

ax_reset = plt.axes([0.8, 0.12, 0.12, 0.03])
btn_reset = Button(ax_reset, 'Reset')

def update(val):
    """Динамічне оновлення графіків (Пункт 3, 5)."""
    amp = s_amp.val
    freq = s_freq.val
    phase = s_phase.val
    mean = s_mean.val
    cov = s_cov.val
    cutoff = s_cutoff.val
    show_noise = check.get_status()[0]
    
    pure_data, noisy_data = harmonic_with_noise(amp, freq, phase, mean, cov, show_noise)
    filtered_data = apply_filter(noisy_data, cutoff, fs)
    
    line_pure.set_ydata(pure_data)
    line_filtered.set_ydata(filtered_data)
    
    if show_noise:
        line_noisy.set_ydata(noisy_data)
        line_noisy.set_visible(True)
    else:
        line_noisy.set_visible(False)
        
    fig.canvas.draw_idle()

s_amp.on_changed(update)
s_freq.on_changed(update)
s_phase.on_changed(update)
s_mean.on_changed(update)
s_cov.on_changed(update)
s_cutoff.on_changed(update)
check.on_clicked(update)

def reset_values(event):
    s_amp.reset()
    s_freq.reset()
    s_phase.reset()
    s_mean.reset()
    s_cov.reset()
    s_cutoff.reset()
    if not check.get_status()[0]:
        check.set_active(0)

btn_reset.on_clicked(reset_values)

plt.show()